# Compress a matrix with blocktt_MALS

We start with a random 16 × 16 matrix, which contains 256 numbers. We will approximate it using an MPO that stores at most 96 numbers, then repeat with a smaller budget to see how the error changes.

In [1]:
import math
import torch
from tltorch import blocktt_MALS

generator = torch.Generator().manual_seed(42)
A = torch.randn(16, 16, dtype=torch.float64, generator=generator)
A.shape, A.dtype, A.numel()

(torch.Size([16, 16]), torch.float64, 256)

We first choose how to split the matrix dimensions across the MPO cores.

`input_modes` splits the 16 columns into dimensions `(2, 2, 2, 2)`. Similarly, `output_modes` splits the 16 rows. Together, these choices give an MPO with four cores. The algorithm chooses the bond dimensions connecting them.

We allow the cores to store at most 96 numbers in total (37.5% of the original matrix’s storage). By default, the algorithm starts with internal bond dimensions of two. For this example, that requires 48 numbers, leaving room to increase the bond dimensions.

In [2]:
input_modes = (2, 2, 2, 2)
output_modes = (2, 2, 2, 2)
max_parameters = 96

print(f"Budget: {max_parameters} / {A.numel()} = {max_parameters / A.numel():.1%}")
initial_storage = 1 * 4 * 2 + 2 * 4 * 2 + 2 * 4 * 2 + 2 * 4 * 1
print("Default initial storage:", initial_storage)

Budget: 96 / 256 = 37.5%
Default initial storage: 48


`blocktt_MALS` constructs an initial MPO, improves its cores, and tries increasing individual bond dimensions while staying within the budget.

It returns `weight`, the compressed representation, and `info`, a dictionary containing the ranks, storage, approximation error, and stopping reason.

In [3]:
weight, info = blocktt_MALS(
    A,
    input_modes=input_modes,
    output_modes=output_modes,
    max_parameters=max_parameters,
)

`weight.to_matrix()` reconstructs a dense matrix from the MPO. We compare it with `A` using the relative Frobenius error:

$$
\frac{\|A-\text{reconstruction}\|_F}{\|A\|_F}.
$$

The smaller the value, the closer the approximation. Zero means exact reconstruction.

To measure storage, we count the entries in the MPO cores. `weight.numel()` gives the size of the reconstructed dense matrix, so we sum `core.numel()` instead.

In [4]:
reconstruction = weight.to_matrix().detach()
stored = sum(core.numel() for core in weight.factors)
relative_error = (
    torch.linalg.vector_norm(A - reconstruction) / torch.linalg.vector_norm(A)
).item()

assert stored == info["final_mpo_parameters"] <= max_parameters
assert math.isclose(
    relative_error, info["final_relative_weight_error"], rel_tol=1e-12, abs_tol=1e-14,
)

The rank profile lists the bond dimensions, including a fixed value of one at each end. For example, `(1, 2, 2, 2, 1)` describes four cores connected by three internal bonds of dimension two.

We print the initial and final profiles to see which dimensions grew, together with the storage and reconstruction error.

In [5]:
print("Initial ranks:", info["initial_rank_profile"])
print("Final ranks:", weight.rank)
print("Stored entries:", stored)
print(f"Storage ratio: {stored} / {A.numel()} = {stored / A.numel():.4f}")
print(f"Relative Frobenius error: {relative_error:.6f}")
print("Stopping reason:", info["termination_reason"])

Initial ranks: (1, 2, 2, 2, 1)
Final ranks: (1, 3, 3, 3, 1)
Stored entries: 96
Storage ratio: 96 / 256 = 0.3750
Relative Frobenius error: 0.733853
Stopping reason: parameter_budget_blocks_growth


All three internal bond dimensions grew from two to three. The resulting MPO stores 96 numbers and has a relative error of about 0.734. The search stopped because another increase would exceed the budget.

Now we repeat the calculation with a budget of 48, using the same matrix and mode dimensions.

In [6]:
smaller_weight, smaller_info = blocktt_MALS(
    A,
    input_modes=input_modes,
    output_modes=output_modes,
    max_parameters=48,
)

In [7]:
smaller_reconstruction = smaller_weight.to_matrix().detach()
smaller_stored = sum(core.numel() for core in smaller_weight.factors)
smaller_error = (
    torch.linalg.vector_norm(A - smaller_reconstruction) / torch.linalg.vector_norm(A)
).item()

assert smaller_stored == smaller_info["final_mpo_parameters"] <= 48
assert math.isclose(
    smaller_error, smaller_info["final_relative_weight_error"], rel_tol=1e-12, abs_tol=1e-14,
)

In [8]:
rows = [
    (48, smaller_weight.rank, smaller_stored, smaller_stored / A.numel(),
     smaller_error, smaller_info["termination_reason"]),
    (max_parameters, weight.rank, stored, stored / A.numel(),
     relative_error, info["termination_reason"]),
]
print("budget  ranks              stored  stored/dense  relative error  stopping reason")
for budget, ranks, entries, ratio, error, reason in rows:
    print(f"{budget:6d}  {str(ranks):17}  {entries:6d}  {ratio:12.4f}  {error:14.6f}  {reason}")

budget  ranks              stored  stored/dense  relative error  stopping reason
    48  (1, 2, 2, 2, 1)        48        0.1875        0.850570  parameter_budget_blocks_growth
    96  (1, 3, 3, 3, 1)        96        0.3750        0.733853  parameter_budget_blocks_growth


With a budget of 48, the initial MPO already uses all the available storage. The algorithm still improves the cores at fixed bond dimensions, but it cannot increase those dimensions.

The relative error is about 0.851, compared with 0.734 at a budget of 96. In this example, doubling the storage gives a closer approximation, although both errors remain large.

This random matrix illustrates how to use the function and compare results. Compression quality depends on the matrix and its tensorization. The search is approximate, so separate runs with larger budgets are not guaranteed to produce lower errors.